# Settings

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/Islamic_Stories_Project"

LORA_PATH = f"{BASE_DIR}/models/Llama3_8B_Arabic_Stories"
CSV_PATH = f"{BASE_DIR}/LlaMa_output.csv"


for label, p in [
    ("LORA", LORA_PATH),
    ("Output CSV", CSV_PATH),
]:
    print(("✓" if os.path.exists(p) else "✗ MISSING"), label, "->", p)

Mounted at /content/drive
✓ LORA -> /content/drive/MyDrive/Islamic_Stories_Project/models/Llama3_8B_Arabic_Stories
✗ MISSING Output CSV -> /content/drive/MyDrive/Islamic_Stories_Project/LlaMa_output.csv


In [2]:
!pip install -q -U transformers accelerate peft bitsandbytes sentence-transformers pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 39.3 MB/s eta 0:00:00


# Load LlaMa

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


# ------------------------------------------------------------
# Hugging Face authentication (needed for gated LLaMA 3)
# ------------------------------------------------------------
HF_TOKEN = "-"
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)


MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit QLoRA config — identical to your original notebook
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Base model
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
)

# Attach your trained LoRA adapter (no retraining)
ft_model = PeftModel.from_pretrained(base, LORA_PATH).eval()
print("✓ LlaMa + LoRA loaded")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✓ LlaMa + LoRA loaded


# Functions

In [6]:
# ============================================================
# Functions — LlaMa generation + summary + query + CSV saving
# ============================================================

import os
import csv
from datetime import datetime

# نفس system prompt القديم
SYSTEM_PROMPT = "أنت لاما، كاتب قصص عربية للأطفال يركز على القيم الإسلامية والعبرة في النهاية."


# ============================================================
# 1. Story generation
# ============================================================

def generate_story_llama(
    age,
    moral,
    topic,
    place=None,
    end_of_story=None,
    dialogue=None,
    num_characters=None,
    country=None,
    season=None,
    activity=None,
    emotion=None,
    plot_twist=None,
    max_new_tokens=400
):
    """
    Generate Arabic children's story using the loaded LlaMa + LoRA model.
    Uses the same original prompt and generation settings.
    """

    features = [
        f"- عُمر الطفل/الطفلة: {age} سنة",
        f"- القيمة الإسلامية (العبرة): {moral}",
        f"- الموضوع العام للقصة: {topic}",
    ]

    if place:
        features.append(f"- مكان أحداث القصة: {place}")
    if country:
        features.append(f"- الدولة: {country}")
    if season:
        features.append(f"- الفصل: {season}")
    if activity:
        features.append(f"- النشاط الرئيسي في القصة: {activity}")
    if num_characters:
        features.append(f"- عدد الشخصيات الأساسية: {num_characters}")
    if emotion:
        features.append(f"- الشعور العام في القصة: {emotion}")

    if dialogue is not None:
        features.append(
            "- تضمين حوار بين الشخصيات"
            if dialogue
            else "- تقليل الحوار والتركيز على السرد"
        )

    if plot_twist is not None:
        features.append(
            "- تحتوي على حبكة مفاجِئة في النهاية"
            if plot_twist
            else "- بدون حبكة مفاجِئة"
        )

    if end_of_story:
        features.append(f"- شكل نهاية القصة المطلوب: {end_of_story}")

    user_prompt = (
        "أريد منك أن تكتب قصة عربية للأطفال بناءً على المواصفات التالية:\n\n"
        + "\n".join(features)
        + "\n\nشروط مهمة:\n"
        "- استخدم لغة عربية مبسطة وممتعة تناسب الأطفال.\n"
        "- اجعل القصة مترابطة وواضحة.\n"
        "- في النهاية، اكتب سطرًا يبدأ بكلمة: \"العبرة:\" ثم قدّم العبرة بشكل صريح وواضح."
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        },
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(device)

    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
        )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    story = tokenizer.decode(gen_ids, skip_special_tokens=True)

    return story.strip()


# ============================================================
# 2. Extract moral
# ============================================================

def extract_moral(text):
    """
    Extract only the explicit moral line after 'العبرة:'.
    """
    marker = "العبرة:"

    if marker not in text:
        return ""

    after_marker = text.split(marker, 1)[1].strip()
    lines = [line.strip() for line in after_marker.splitlines() if line.strip()]

    if not lines:
        return ""

    moral_line = lines[0]
    moral_line = moral_line.strip().strip('"').strip("“”").strip("«»")

    return moral_line.strip()


# ============================================================
# 3. Summary for retrieval
# ============================================================

def summarize_story_for_retrieval(story_text, topic, moral):
    """
    Generate short value-focused summary using ALLaM.
    Same summary prompt and settings.
    """

    prompt = f"""
لديك قصة عربية للأطفال، وأريد تلخيصها لغرض البحث عن حديث نبوي مناسب.

الموضوع: {topic}
القيمة الإسلامية التي أدخلها المستخدم: {moral}

القصة:
{story_text}

اكتب ملخصًا قصيرًا جدًا من جملة واحدة فقط.
ركّز على:
- الحدث الأساسي في القصة
- القيمة الإسلامية أو الأخلاقية
- العبرة المناسبة

تجنب ذكر التفاصيل الجانبية مثل الأسماء، المكان، الفصل، أو الوصف الطويل.
لا تكتب عنوانًا ولا شرحًا. اكتب الملخص فقط.
""".strip()

    messages = [
        {
            "role": "system",
            "content": "أنت مساعد يلخص قصص الأطفال العربية لغرض مطابقة القصة مع حديث نبوي مناسب."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(device)

    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    summary = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    summary = summary.replace("الملخص:", "").replace("ملخص:", "").strip()
    summary = summary.strip().strip('"').strip("“”").strip("«»")

    lines = [line.strip() for line in summary.splitlines() if line.strip()]
    if lines:
        summary = lines[0]

    return summary.strip()


# ============================================================
# 4. Query builders
# ============================================================

def build_query_with_summary(topic, moral, story_summary=None):
    """
    Query with summary.
    The summary is stored inside the query only.
    """
    q = (
        f"القيمة الإسلامية: {topic}\n"
        f"العبرة من القصة: {moral}"
    )

    if story_summary:
        q += f"\nملخص القصة: {story_summary}"

    return q


def build_query_no_summary(topic, moral):
    """
    Query without summary.
    """
    return (
        f"القيمة الإسلامية: {topic}\n"
        f"العبرة من القصة: {moral}"
    )


# ============================================================
# 5. CSV saving
# ============================================================

GEN_COLUMNS = [
    "sample_id",
    "timestamp",

    "model_name",

    "input_age",
    "input_topic",
    "input_moral",
    "input_place",
    "input_country",
    "input_season",
    "input_activity",
    "input_emotion",
    "input_dialogue",
    "input_plot_twist",
    "input_end_of_story",

    "generated_story",
    "extracted_moral",

    "query_with_summary",
    "query_no_summary",
]


def append_generation_csv(path, row):
    """
    Append one generation result to CSV.
    """
    new_file = not os.path.exists(path)

    parent = os.path.dirname(os.path.abspath(path))
    if parent:
        os.makedirs(parent, exist_ok=True)

    with open(path, "w" if new_file else "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=GEN_COLUMNS)

        if new_file:
            writer.writeheader()

        writer.writerow(row)

    return row


def get_next_sample_id(path):
    """
    Auto sample_id based on existing CSV rows.
    """
    if os.path.exists(path):
        try:
            old_df = pd.read_csv(path)
            return len(old_df) + 1
        except Exception:
            return 1

    return 1


print("✓ LlaMa generation functions ready")

✓ LlaMa generation functions ready


## أمثلة

مثال الصدق

In [10]:
# ============================================================
# Run Llama generation + summary + queries + save
# ============================================================

sample_id = get_next_sample_id(CSV_PATH)

inp = {
    "age": 8,
    "topic": "الصدق",
    "moral": "أهمية الصدق",
    "place": "المدرسة",
    "country": "السعودية",
    "season": "الخريف",
    "activity": "وقت اللعب",
    "emotion": "حماسي",
    "dialogue": True,
    "plot_twist": True,
    "end_of_story": "اعتراف صادق",
    "num_characters": None,
}

print("=" * 80)
print("Running LlaMa generation")
print("Sample ID:", sample_id)
print("=" * 80)

story = generate_story_llama(
    age=inp["age"],
    moral=inp["moral"],
    topic=inp["topic"],
    place=inp.get("place"),
    country=inp.get("country"),
    season=inp.get("season"),
    activity=inp.get("activity"),
    emotion=inp.get("emotion"),
    dialogue=inp.get("dialogue"),
    plot_twist=inp.get("plot_twist"),
    end_of_story=inp.get("end_of_story"),
    num_characters=inp.get("num_characters"),
    max_new_tokens=400,
)

extracted_moral = extract_moral(story)

story_summary = summarize_story_for_retrieval(
    story_text=story,
    topic=inp["topic"],
    moral=inp["moral"]
)

query_with_summary = build_query_with_summary(
    topic=inp["topic"],
    moral=inp["moral"],
    story_summary=story_summary
)

query_no_summary = build_query_no_summary(
    topic=inp["topic"],
    moral=inp["moral"]
)

out_row = {
    "sample_id": sample_id,
    "timestamp": datetime.now().isoformat(timespec="seconds"),

    "model_name": "Llama",

    "input_age": inp.get("age"),
    "input_topic": inp.get("topic"),
    "input_moral": inp.get("moral"),
    "input_place": inp.get("place"),
    "input_country": inp.get("country"),
    "input_season": inp.get("season"),
    "input_activity": inp.get("activity"),
    "input_emotion": inp.get("emotion"),
    "input_dialogue": inp.get("dialogue"),
    "input_plot_twist": inp.get("plot_twist"),
    "input_end_of_story": inp.get("end_of_story"),

    "generated_story": story,
    "extracted_moral": extracted_moral,

    "query_with_summary": query_with_summary,
    "query_no_summary": query_no_summary,
}

append_generation_csv(CSV_PATH, out_row)

print("\n" + "=" * 80)
print("Generated Story")
print("=" * 80)
print(story)

print("\n" + "=" * 80)
print("Extracted Moral")
print("=" * 80)
print(extracted_moral)

print("\n" + "=" * 80)
print("Query WITH summary")
print("=" * 80)
print(query_with_summary)

print("\n" + "=" * 80)
print("Query WITHOUT summary")
print("=" * 80)
print(query_no_summary)

print("\nSaved to:")
print(CSV_PATH)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running LlaMa generation
Sample ID: 1


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Story
في يوم جميل من أيام الخريف في مدينة الرياض الساحرة، كان الأطفال يتشاركون في وقت اللعب بعد المدرسة في حديقة المدرسة الكبيرة. بينهم، كان هناك طفل يدعى سلمان، عمره 8 سنوات، ويحب اللعب مع أصدقائه دائماً. 

ذات يوم، قرر سلمان وأصدقاؤه لعبوا لعبة سرية، حيث يجب أن يكون كل منهم على علم باللعبة وألا يقلق أو يعاتب أحد إذا لم يكن على ما يرام. لكن سلمان، الذي كان يحلم بأن يصبح قائدًا في اللعبة، قرر أن يلعب وحدها دون أن يخبر أصدقاؤه، متشككًا في قدرتة على تحقيق النجاح وحدها.

في البداية، كان كل شيء يبدو على ما يرام. سلمان، بكل حنق ووخامة، كان يلعب اللعبة بكل قوته، ولكن في النهاية، بدأت المشاكل تظهر. لم يكن يستطيع حل كل مشكلة وحدها، وكان يتضايق ويلوم نفسه أكثر وأكثر.

في يوم اللعب التالي، التقى سلمان بصدقائه، الذي كانوا يبحثون عنه. سألهم، باختصار، عن أين يلعب. لكن سلمان، الذي كان يشعر بالذنب بسبب خيانته، لم يضفيهما في اللعبة، وبدلاً من ذلك، سمح لهم باللعب جنبًا إلى جنب معه.

في اللعبة، بدأ الأطفال يعملون معًا، وبدلاً من أن تكون اللعبة صعبة، أصبحت أكثر تفاعلاً ودر

Extracted Moral


Q

مثال الصيام

In [13]:
# ============================================================
# Run LlaMa generation + summary + queries + save
# ============================================================

sample_id = get_next_sample_id(CSV_PATH)

inp = {
    "age": 8,
    "topic": "الصيام",
    "moral": "فضل الصيام",
    "place": "المنزل",
    "country": "السعودية",
    "season": "الشتاء",
    "activity": "الاستعداد لشهر رمضان",
    "emotion": "روحاني",
    "dialogue": True,
    "plot_twist": False,
    "end_of_story": "طمأنينة",
    "num_characters": None,
}

print("=" * 80)
print("Running Llama generation")
print("Sample ID:", sample_id)
print("=" * 80)

story = generate_story_llama(
    age=inp["age"],
    moral=inp["moral"],
    topic=inp["topic"],
    place=inp.get("place"),
    country=inp.get("country"),
    season=inp.get("season"),
    activity=inp.get("activity"),
    emotion=inp.get("emotion"),
    dialogue=inp.get("dialogue"),
    plot_twist=inp.get("plot_twist"),
    end_of_story=inp.get("end_of_story"),
    num_characters=inp.get("num_characters"),
    max_new_tokens=400,
)

extracted_moral = extract_moral(story)

story_summary = summarize_story_for_retrieval(
    story_text=story,
    topic=inp["topic"],
    moral=inp["moral"]
)

query_with_summary = build_query_with_summary(
    topic=inp["topic"],
    moral=inp["moral"],
    story_summary=story_summary
)

query_no_summary = build_query_no_summary(
    topic=inp["topic"],
    moral=inp["moral"]
)

out_row = {
    "sample_id": sample_id,
    "timestamp": datetime.now().isoformat(timespec="seconds"),

    "model_name": "Llama",

    "input_age": inp.get("age"),
    "input_topic": inp.get("topic"),
    "input_moral": inp.get("moral"),
    "input_place": inp.get("place"),
    "input_country": inp.get("country"),
    "input_season": inp.get("season"),
    "input_activity": inp.get("activity"),
    "input_emotion": inp.get("emotion"),
    "input_dialogue": inp.get("dialogue"),
    "input_plot_twist": inp.get("plot_twist"),
    "input_end_of_story": inp.get("end_of_story"),

    "generated_story": story,
    "extracted_moral": extracted_moral,

    "query_with_summary": query_with_summary,
    "query_no_summary": query_no_summary,
}

append_generation_csv(CSV_PATH, out_row)

print("\n" + "=" * 80)
print("Generated Story")
print("=" * 80)
print(story)

print("\n" + "=" * 80)
print("Extracted Moral")
print("=" * 80)
print(extracted_moral)

print("\n" + "=" * 80)
print("Query WITH summary")
print("=" * 80)
print(query_with_summary)

print("\n" + "=" * 80)
print("Query WITHOUT summary")
print("=" * 80)
print(query_no_summary)

print("\nSaved to:")
print(CSV_PATH)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running Llama generation
Sample ID: 2


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Story
في يوم بارد وجميل من أيام الشتاء في المملكة العربية السعودية، كان الطفل أمجد، عمره 8 سنوات، يجلس في حديقته الوردية الجميلة مع والده، يبتسمان بكل سعادة. أمجد كان يفكر في صيام شهر رمضان القادم، وكان يرغب في أن يكون جاهزًا للاستعداد قبل أن يأتي الشهر الكريم.

قال أمجد، بصوته المليء بالفضول، "أبوي، أريد أن أكون جاهزًا لصيام رمضان قبل شهر، أردت أن أفعل كثيرون من الأشياء الجيدة."

قال والده، بابتسامة حزينة، "أنت بالفعل جيد، يا أمجد، لكن لمن الأفضل أن نقوم بالاستعداد جسورًا وأنت تعلمين قيمة الصيام؟"

نظر أمجد إلى والده بفضول، ثم قال، "لم أدرك قيمة الصيام بعد، هل تعلم شيئًا عنها؟"

ابتسم والده، وقال، "نعم، الصيام هو فضل كبير. عندما نصوم، نتعلم الابتعاد عن الأشياء التي ليس لها قيمة حقيقية في الحياة، ونحن نركز على ما يجعلهم يصلون ويذرون. ويجلب الصيام الرحمة والبركة."

عرف أمجد قيمة الصيام، فبدأ في أن يفكر في الأشياء التي يجب أن يتركها ويقوم بما هو أفضل. بدأ بالتعلم والاستعداد لشهر رمضان.

في يوم الشتاء الجميل، عندما كان الصيام قريبًا، شعر أمجد بالتأثر العميق من قوله والده. فقرر


مثال الصلاة

In [14]:
# ============================================================
# Run LlaMa generation + summary + queries + save
# ============================================================

sample_id = get_next_sample_id(CSV_PATH)

inp = {
    "age": 8,
    "topic": "الصلاة",
    "moral": "أهمية الصلاة",
    "place": "المسجد",
    "country": "السعودية",
    "season": "الصيف",
    "activity": "الذهاب للمسجد",
    "emotion": "حنون",
    "dialogue": True,
    "plot_twist": False,
    "end_of_story": "نهاية سعيدة",
    "num_characters": None,
}

print("=" * 80)
print("Running Llama generation")
print("Sample ID:", sample_id)
print("=" * 80)

story = generate_story_llama(
    age=inp["age"],
    moral=inp["moral"],
    topic=inp["topic"],
    place=inp.get("place"),
    country=inp.get("country"),
    season=inp.get("season"),
    activity=inp.get("activity"),
    emotion=inp.get("emotion"),
    dialogue=inp.get("dialogue"),
    plot_twist=inp.get("plot_twist"),
    end_of_story=inp.get("end_of_story"),
    num_characters=inp.get("num_characters"),
    max_new_tokens=400,
)

extracted_moral = extract_moral(story)

story_summary = summarize_story_for_retrieval(
    story_text=story,
    topic=inp["topic"],
    moral=inp["moral"]
)

query_with_summary = build_query_with_summary(
    topic=inp["topic"],
    moral=inp["moral"],
    story_summary=story_summary
)

query_no_summary = build_query_no_summary(
    topic=inp["topic"],
    moral=inp["moral"]
)

out_row = {
    "sample_id": sample_id,
    "timestamp": datetime.now().isoformat(timespec="seconds"),

    "model_name": "llama",

    "input_age": inp.get("age"),
    "input_topic": inp.get("topic"),
    "input_moral": inp.get("moral"),
    "input_place": inp.get("place"),
    "input_country": inp.get("country"),
    "input_season": inp.get("season"),
    "input_activity": inp.get("activity"),
    "input_emotion": inp.get("emotion"),
    "input_dialogue": inp.get("dialogue"),
    "input_plot_twist": inp.get("plot_twist"),
    "input_end_of_story": inp.get("end_of_story"),

    "generated_story": story,
    "extracted_moral": extracted_moral,

    "query_with_summary": query_with_summary,
    "query_no_summary": query_no_summary,
}

append_generation_csv(CSV_PATH, out_row)

print("\n" + "=" * 80)
print("Generated Story")
print("=" * 80)
print(story)

print("\n" + "=" * 80)
print("Extracted Moral")
print("=" * 80)
print(extracted_moral)

print("\n" + "=" * 80)
print("Query WITH summary")
print("=" * 80)
print(query_with_summary)

print("\n" + "=" * 80)
print("Query WITHOUT summary")
print("=" * 80)
print(query_no_summary)

print("\nSaved to:")
print(CSV_PATH)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running Llama generation
Sample ID: 3


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Story
في يوم صيفي مشرق، كان يعيش في بلدنا الجميل، السعودية، طفل صغير يدعى يوسف. كان يوسف يبلغ من العمر 8 سنوات ويتعلم في فصل الصيف، بينما كان يحب ممارسة النشاطات الخارجية مع أصدقائه في الصباح.

في أحد الأيام، قرر يوسف أن يذهب مع أبيه إلى المسجد لأداء الصلاة. كان يوسف يفكر دائمًا في الصلاة وعظمتها في قلبه، فقرر أن يغتنم الفرصة ليتعلم كيفية أداء الصلاة الصحيحة.

"أبي، اليوم أنا قررت أن أذهب معك إلى المسجد، وأدعو الله أن يعطيني القوة والشجاعة في أي مهمة،" قال يوسف بصوت مليئ بالفرح.

"مرحباً يوسف، هذا فعل جيد، وسنذهب معًا. في المسجد، ستنوبنا الصلاة عنا، وسنغتنم الفرصة لنتعلم كيفية أداء الصلاة الصحيحة،" أجاب الأب بخفض صوته.

ذات صباح، قام يوسف والأب بالذهاب إلى المسجد. كانت الحرارة عالية، ولكن يوسف لم يشعر بالملل، بل كان يمتلئ بالشغف والدفء. عندما وصلوا إلى المسجد، وقفوا أمام المئذنة، وبدأ يوسف الأب بالتحدث عن أهمية الصلاة في حياة المسلمين.

"الصلاة، يا يوسف، هي طاعة الله وخدمة لنا، ويحفظنا الصلاة من الشر ويرحمنا. يجب أن نلتزمها كل يوم، مهما كانت الظروف،" قال الأب.

Extracted Mor

In [15]:
import os
import pandas as pd

print("CSV_PATH:", CSV_PATH)
print("Exists:", os.path.exists(CSV_PATH))

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print("Rows:", len(df))
    display(df.tail())
else:
    print("File does not exist yet.")

CSV_PATH: /content/drive/MyDrive/Islamic_Stories_Project/LlaMa_output.csv
Exists: True
Rows: 3


,sample_id,timestamp,model_name,input_age,input_topic,input_moral,input_place,input_country,input_season,input_activity,input_emotion,input_dialogue,input_plot_twist,input_end_of_story,generated_story,extracted_moral,query_with_summary,query_no_summary
0,1,2026-05-22T04:52:43,Llama,8,الصدق,أهمية الصدق,المدرسة,السعودية,الخريف,وقت اللعب,حماسي,True,True,اعتراف صادق,في يوم جميل من أيام الخريف في مدينة الرياض الس...,NaN,القيمة الإسلامية: الصدق\nالعبرة من القصة: أهمي...,القيمة الإسلامية: الصدق\nالعبرة من القصة: أهمي...
1,2,2026-05-22T04:55:29,Llama,8,الصيام,فضل الصيام,المنزل,السعودية,الشتاء,الاستعداد لشهر رمضان,روحاني,True,False,طمأنينة,في يوم بارد وجميل من أيام الشتاء في المملكة ال...,NaN,القيمة الإسلامية: الصيام\nالعبرة من القصة: فضل...,القيمة الإسلامية: الصيام\nالعبرة من القصة: فضل...
2,3,2026-05-22T04:58:19,llama,8,الصلاة,أهمية الصلاة,المسجد,السعودية,الصيف,الذهاب للمسجد,حنون,True,False,نهاية سعيدة,في يوم صيفي مشرق، كان يعيش في بلدنا الجميل، ال...,NaN,القيمة الإسلامية: الصلاة\nالعبرة من القصة: أهم...,القيمة الإسلامية: الصلاة\nالعبرة من القصة: أهم...
